# ResNet على بيانات MNIST (نسخة مبسطة)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# تحميل بيانات MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# تطبيع القيم من 0-255 الى 0-1
x_train = x_train / 255.0
x_test = x_test / 255.0

# اضافة بعد القناة (channel) عشان الصور تبقى (28,28,1)
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)


## بناء الموديل (مع Residual Block بسيط)

In [ ]:
from tensorflow.keras import Input, Model

def residual_block(x, filters):
    shortcut = x
    y = layers.Conv2D(filters, (3,3), padding="same", activation="relu")(x)
    y = layers.Conv2D(filters, (3,3), padding="same")(y)
    # لو عدد الفلاتر مختلف نظبط الـ shortcut بنفس الحجم
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1,1), padding="same")(shortcut)
    out = layers.Add()([y, shortcut])
    out = layers.Activation("relu")(out)
    return out

inputs = Input(shape=(28,28,1))
x = layers.Conv2D(32, (3,3), activation="relu", padding="same")(inputs)
x = residual_block(x, 32)
x = layers.MaxPooling2D()(x)
x = residual_block(x, 64)
x = layers.MaxPooling2D()(x)
x = layers.Flatten()(x)
x = layers.Dense(128, activation="relu")(x)
outputs = layers.Dense(10, activation="softmax")(x)

model = Model(inputs, outputs)
model.summary()


## التدريب

In [ ]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

model.fit(x_train, y_train, epochs=5, batch_size=128, validation_split=0.1)


## التقييم

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_acc)
